## 로지스틱 회귀

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

# 유방암 데이터셋 로드
cancer = load_breast_cancer()

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# StandardScaler로 feature를 표준화
# (평균 0, 분산 1로 맞춰 로지스틱 회귀 성능 안정화)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(cancer.data)

# 학습용 / 테스트용 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    data_scaled,
    cancer.target,
    test_size=0.3,
    random_state=0
)

In [10]:
from sklearn.metrics import accuracy_score, roc_auc_score

# 로지스틱 회귀 모델 생성 및 학습
# solver를 따로 지정하지 않으면 기본값은 'lbfgs'
lr_clf = LogisticRegression()
lr_clf.fit(X_train, y_train)

# 테스트 데이터 예측값(클래스) 생성
lr_preds = lr_clf.predict(X_test)

# 테스트 데이터 예측 확률값 생성
# [:, 1]은 양성 클래스(1)일 확률만 추출
lr_preds_proba = lr_clf.predict_proba(X_test)[:, 1]

# accuracy와 ROC-AUC 평가
print(
    'accuracy: {0:.3f}, roc_auc:{1:.3f}'.format(
        accuracy_score(y_test, lr_preds),
        roc_auc_score(y_test, lr_preds_proba)
    )
)

accuracy: 0.977, roc_auc:0.995


In [11]:
# 여러 solver별로 로지스틱 회귀 성능 비교
solvers = ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']

for solver in solvers:
    # solver별 모델 생성
    # 일부 solver는 수렴이 느릴 수 있어 max_iter를 600으로 증가
    lr_clf = LogisticRegression(solver=solver, max_iter=600)

    # 학습
    lr_clf.fit(X_train, y_train)

    # 예측 클래스 / 예측 확률 계산
    lr_preds = lr_clf.predict(X_test)
    lr_preds_proba = lr_clf.predict_proba(X_test)[:, 1]

    # 성능 출력
    print(
        'solver:{0}, accuracy: {1:.3f}, roc_auc:{2:.3f}'.format(
            solver,
            accuracy_score(y_test, lr_preds),
            roc_auc_score(y_test, lr_preds_proba)
        )
    )

solver:lbfgs, accuracy: 0.977, roc_auc:0.995
solver:liblinear, accuracy: 0.982, roc_auc:0.995
solver:newton-cg, accuracy: 0.977, roc_auc:0.995
solver:sag, accuracy: 0.982, roc_auc:0.995
solver:saga, accuracy: 0.982, roc_auc:0.995


In [12]:
from sklearn.model_selection import GridSearchCV

# 하이퍼파라미터 후보 설정
# solver와 penalty 조합에 따라 일부 경우는 지원되지 않아 경고가 발생할 수 있음
params = {
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l2', 'l1'],
    'C': [0.01, 0.1, 1, 5, 10]
}

# 로지스틱 회귀 모델 생성
lr_clf = LogisticRegression()

# GridSearchCV를 이용해 최적 하이퍼파라미터 탐색
# 평가 지표는 accuracy, 교차검증은 3-fold 사용
grid_clf = GridSearchCV(
    lr_clf,
    param_grid=params,
    scoring='accuracy',
    cv=3
)

# 전체 표준화 데이터로 학습
grid_clf.fit(data_scaled, cancer.target)

# 최적 파라미터와 최고 평균 정확도 출력
print(
    '최적 하이퍼 파라미터:{0}, 최적 평균 정확도:{1:.3f}'.format(
        grid_clf.best_params_,
        grid_clf.best_score_
    )
)

최적 하이퍼 파라미터:{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}, 최적 평균 정확도:0.979


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
15 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_solver